#Retrieval Part

Installing imortant Libraries

In [4]:
#!pip install pypdf sentence-transformers faiss-cpu
#!pip install -U langchain-experimental langchain-community
#!pip install -U langchain-huggingface

Important Imports

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings
from sentence_transformers import SentenceTransformer
from google.colab import drive
from pypdf import PdfReader
import faiss
import os

/tmp/ipykernel_678/3952169476.py:2: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


In [6]:
drive.mount('/content/drive')

Mounted at /content/drive


Getting a pdf Document

In [7]:
pdf_path='/content/drive/MyDrive/machine_learning_basics_rag.pdf'
pdf_file=os.path.exists(pdf_path)
print(pdf_file ) #Checking if the path Exist

True


Extracting text

In [8]:

reader=PdfReader(pdf_path)

text=''

for i in reader.pages:
  extract_text=i.extract_text()
  if extract_text:
    text+=extract_text+'\n'


print(text[:1498])


Machine Learning Basics
 A Simple Reference Document for a Retrieval-Augmented Generation (RAG) Project
1. What is Machine Learning?
Machine Learning (ML) is a branch of artificial intelligence that allows computers to learn patterns from
data and use those patterns to make predictions or decisions. Instead of explicitly programming every
rule, we provide examples and allow a model to learn a relationship between inputs and outputs.
For example, a model can be trained using historical house data containing features such as area,
number of rooms, and location. After learning from the examples, the model can estimate the price of
a new house.
2. Types of Machine Learning
Machine learning is commonly divided into supervised learning, unsupervised learning, and
reinforcement learning. The main difference is the type of information available during learning.
2.1 Supervised Learning
In supervised learning, the training data contains both input features and known target labels. The
model lear

Chunking (easy overlapping)

In [9]:
def chunk_text(text,chunk_size=500,overlap=20):
  chunks=[]
  start=0
  end=0
  while start<len(text):
    end=start+chunk_size
    if end+overlap<len(text):
      chunks.append(text[start:end])
      start=end-overlap
    else:
      chunks.append(text[start:])
      break
  return chunks

13 Chunks Made
Context is clearly lost

In [10]:
my_chunks=chunk_text(text)
print(len(my_chunks))
#print(my_chunks[0]) #Uncomment to see Chunk Example

13


##Lets see Langchain Chunking

Recusrive Chunking

In [11]:
splitter=RecursiveCharacterTextSplitter()
rec_chunks=splitter.split_text(text)

In [12]:
#rec_chunks[0] #Uncomment to see Chunk Example
len(rec_chunks)   #only 2 chunks so the llm has to remember a lot of info- also there has to be unnecessary info in each chunk

2

Semantic Chunking-Requires Text to be Embedded

In [13]:
embeddings=HuggingFaceEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2")
splitter=SemanticChunker(
    embeddings=embeddings

)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [14]:
sem_chunk=splitter.split_text(text)

This is the best type context is preserved

In [15]:
print(len(sem_chunk))
#print(sem_chunk[0]) #Uncomment to see Chunk

5


Now we have to embedd our text

In [16]:
embeddings=SentenceTransformer("sentence-transformers/all-MiniLM-L6-V2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Using Semantic Chunking for embedding my text

In [17]:
chunk_embedding=embeddings.encode(
    sem_chunk,
    convert_to_numpy=True
)

In [18]:
chunk_embedding.shape
# 5 Chunks of 384 Dimensions

(5, 384)

Setting up a Vector DB

In [83]:
dimensions=chunk_embedding.shape[1]
index=faiss.IndexFlatL2(dimensions)
index.add(chunk_embedding)


lets suppose a user comes

In [84]:
questions='Does unsupervised learning use target labels'
question_embedding=embeddings.encode( #Embedd user Question
    [questions],
    convert_to_numpy=True
)


In [85]:
question_embedding.shape

(1, 384)

In [87]:
dist,indices=index.search(
    question_embedding,
    k=2
)

In [88]:
def get_chunk(indices):
  final_chunks=[]
  for i in indices[0]:
    final_chunks.append(sem_chunk[i])
  return final_chunks


In [89]:
retreived_chunks=get_chunk(indices)

In [90]:
context='\n\n'.join(retreived_chunks)

In [91]:
#context

'Machine Learning Basics\n A Simple Reference Document for a Retrieval-Augmented Generation (RAG) Project\n1. What is Machine Learning? Machine Learning (ML) is a branch of artificial intelligence that allows computers to learn patterns from\ndata and use those patterns to make predictions or decisions. Instead of explicitly programming every\nrule, we provide examples and allow a model to learn a relationship between inputs and outputs. For example, a model can be trained using historical house data containing features such as area,\nnumber of rooms, and location. After learning from the examples, the model can estimate the price of\na new house. 2. Types of Machine Learning\nMachine learning is commonly divided into supervised learning, unsupervised learning, and\nreinforcement learning. The main difference is the type of information available during learning. 2.1 Supervised Learning\nIn supervised learning, the training data contains both input features and known target labels. The\

#Generation Part

In [75]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
model_name='Qwen/Qwen2.5-1.5B-Instruct'
tokenizer=AutoTokenizer.from_pretrained(model_name)
model=AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map='auto'
                                           )

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Wtring a prompt for the LLM

In [92]:
prompt = f"""
You are a helpful assistant answering questions using the provided context.

Answer the question using ONLY the information in the context.

If the answer cannot be found in the context, say:
"I don't have enough information in the provided documents."

Context:
{context}

Question:
{questions}

Answer:
"""

In [94]:
input=tokenizer(prompt,return_tensors='pt').to(model.device)

In [95]:
with torch.no_grad():
  outputs=model.generate(
    **input,
    max_new_tokens=512,
    temperature=0.1
  )

Number of tokens in the OG prompt

In [97]:
input_len=input['input_ids'].shape[1]

In [98]:
generated_info=tokenizer.decode(
    outputs[0][input_len:],
    skip_special_tokens=True
)


#Final Results

In [99]:
def result(questions,generated_info):
  print(f'Question: {questions}\n')
  print(f'Answer: {generated_info}')

In [100]:
result(questions,generated_info)

Question: Does unsupervised learning use target labels

Answer: No, unsupervised learning does not use target labels. In unsupervised learning, the goal is to discover patterns or structures within the data without any predefined targets or labels. The algorithm tries to find meaningful representations or clusters within the dataset itself. This contrasts with supervised learning, where there are labeled examples to guide the learning process. Therefore, unsupervised learning focuses on finding intrinsic properties of the data rather than making predictions about specific outcomes.
